# Visual Caption Generation (Qwen2.5-VL)

Generates a detailed caption for every figure MinerU2.5 extracted from a
PDF — photos, labeled diagrams, charts, and tables all go through
**Qwen2.5-VL**, since the goal here is a detailed, well-grounded
description rather than a clinical/diagnostic reading.

For diagrams MinerU2.5 tags as `sub_type == "text_image"`, it also OCRs
the leader-line labels into the `content` field. Those labels are passed
into the prompt as grounding context, so the model transcribes and places
the existing labels instead of re-reading the image from scratch (this is
what previously tripped up a photo-specialized model into hallucinating a
fake CT scan reading for a labeled mouth-anatomy diagram).

**Hardware note:** runs on Apple Silicon (MPS, no CUDA). Uses bfloat16 and
loads to CPU before `.to("mps")` — torch's MPS backend has unresolved
SIGSEGVs in its fp16 cast kernel and in device_map-based loading (see
`src/captioning/qwen_vl.py` docstring for issue links).

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from captioning.qwen_vl import QwenVLCaptioner

PROJECT_ROOT

/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student')

## Load MinerU2.5 output for one PDF

In [2]:
PDF_STEM = "Anatomy of Face and Oral Cavity - Basic of DEMN.pdf"
OUTPUT_DIR = PROJECT_ROOT / "output" / PDF_STEM / "hybrid_auto"
CONTENT_LIST_PATH = OUTPUT_DIR / f"{PDF_STEM}_content_list.clean.json"

with open(CONTENT_LIST_PATH) as f:
    content_list = json.load(f)

visual_items = [
    item for item in content_list
    if item.get("type") in {"image", "chart", "table", "diagram"} and item.get("img_path")
]
len(visual_items), visual_items[0]

(7,
 {'type': 'table',
  'img_path': 'images/7ec38269bad8a2ec4e458cf59d33338298ede388357aed611c0ad2acef968242.jpg',
  'table_caption': [],
  'table_footnote': [],
  'table_body': '<table><tr><td>Single bones</td><td>Paired bones</td></tr><tr><td>Vomer</td><td>Maxillary</td></tr><tr><td>Mandible</td><td>Palatine</td></tr><tr><td></td><td>Zygomatic</td></tr><tr><td></td><td>Lacrimal</td></tr><tr><td></td><td>Nasal</td></tr><tr><td></td><td>Inferior nasal conchae</td></tr></table>',
  'bbox': [318, 703, 561, 975],
  'page_idx': 5})

## Smoke test on one labeled diagram

In [3]:
captioner = QwenVLCaptioner()

# Prefer a text_image (labeled diagram) sample if this PDF has one.
sample = next((i for i in visual_items if i.get("sub_type") == "text_image"), visual_items[0])

image_path = OUTPUT_DIR / sample["img_path"]
caption = captioner.caption(str(image_path), labels=sample.get("content") or None)

print(json.dumps({
    "image_path": str(image_path),
    "content_type": sample["type"],
    "sub_type": sample.get("sub_type"),
    "caption": caption,
}, indent=2))

Loading weights: 100%|██████████| 729/729 [00:00<00:00, 10638.29it/s]


{
  "image_path": "/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf/hybrid_auto/images/1d9fe1e0fa79ba9feee84909a4ba81cf980f72d6e78a94153cda1ce31dce24ce.jpg",
  "content_type": "image",
  "sub_type": "text_image",
  "caption": "The image is a detailed anatomical illustration of the muscles of the head and neck. The muscles are labeled with text pointing to specific areas of the head and neck. Here's a breakdown of the labels and their positions:\n\n1. **frontalis** - This label points to the forehead area.\n2. **temporalis** - This label points to the temporal region, which includes the muscles on the side of the head near the temple.\n3. **occipitalis** - This label points to the occipital region at the back of the head.\n4. **orbicularis oculi** - This label points to the circular muscle around the eye.\n5. **orbicularis oris** - This label points to the circular muscle around th

## Batch run over all visuals in the PDF

In [4]:
results = []

for item in visual_items:
    image_path = OUTPUT_DIR / item["img_path"]
    caption = captioner.caption(str(image_path), labels=item.get("content") or None)
    results.append({
        "image_path": str(image_path),
        "content_type": item["type"],
        "sub_type": item.get("sub_type"),
        "caption": caption,
    })

captioner.unload()
len(results)

7

In [5]:
results_path = OUTPUT_DIR / f"{PDF_STEM}_stage1_captions.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

results_path

PosixPath('/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf/hybrid_auto/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_stage1_captions.json')